In [ ]:
import pandas as pd
import sys
import os
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
import gc
sys.path.append(os.path.abspath(".."))

import importlib
import src.preprocessing_data.pre_processing as pre_processing_module
importlib.reload(pre_processing_module)

from src.preprocessing_data.pre_processing import PreProcessor


#CONSTANTS
RANDOM_STATE = 42
CHUNK_SIZE = 250000 #DON'T MAKE THIS LARGE
TARGET_POS = 7000000
TARGET_NEG = 3000000
SAMPLE_RATE = 0.50

filepath = "../src/data/raw/all_reviews.csv"
output_path = "../src/data/interim/balanced_10m_reviews.csv"


#preprocessing
def preprocess_chunk(chunk, filepath):
    """
    Apply cleaning and filtering pipeline to a chunk of data.
    """
    p = PreProcessor(filepath)
    p.df = chunk

    p.remove_uninteresting_columns()
    p.filter_english_reviews()
    p.remove_no_playtime()
    p.remove_playtime_outliers()
    p.clean_review_text()
    p.remove_empty_reviews()
    p.handle_missing_numeric_values()
    p.handle_missing_target_variable()

    return p.df


def split_by_label(df):
    """
    Split dataframe into positive and negative classes.
    """
    return df[df["voted_up"]], df[~df["voted_up"]]


#chunk pipeline
pos_chunks = []
neg_chunks = []
total_rows = 0

print("Phase 1: scanning, cleaning, and sampling")

for chunk in pd.read_csv(filepath, chunksize=CHUNK_SIZE):
    total_rows += len(chunk)

    df = preprocess_chunk(chunk, filepath)

    if df.empty:
        continue

    sampled = df.sample(frac=SAMPLE_RATE, random_state=RANDOM_STATE)

    pos, neg = split_by_label(sampled)
    pos_chunks.append(pos)
    neg_chunks.append(neg)

    print(f"Processed {total_rows:,} rows", end="\r")


print("Scanning complete, now balancing.")


#balance samples
all_pos = pd.concat(pos_chunks, ignore_index=True)
all_neg = pd.concat(neg_chunks, ignore_index=True)

print(f"Available samples - Positive: {len(all_pos):,}, Negative: {len(all_neg):,}\n")

pos_final = all_pos.sample(n=min(TARGET_POS, len(all_pos)), random_state=RANDOM_STATE)
neg_final = all_neg.sample(n=min(TARGET_NEG, len(all_neg)), random_state=RANDOM_STATE)

final_df = (pd.concat([pos_final, neg_final], ignore_index=True).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True))


#output
final_df.to_csv(output_path, index=False)

print(f"Saved dataset to: {output_path}")
print(f"Final count: {len(final_df):,} rows")
print(final_df["voted_up"].value_counts())

In [ ]:
#CONSTANTS

CHUNK_SIZE = 100000  #DON'T MAKE THIS LARGE
MAX_FEATURES = 1000

INPUT_FILE = "../src/data/interim/balanced_10m_reviews.csv"
OUTPUT_FILE = "../src/data/interim/balanced_tfidf_10m.csv"

sample = pd.read_csv(INPUT_FILE, nrows=500000, usecols=['review'])
vectorizer = TfidfVectorizer(max_features=MAX_FEATURES, stop_words='english')
vectorizer.fit(sample['review'].fillna(''))

#feature creation for tf idf per term
tfidf_cols = [f"tfidf_{w}" for w in vectorizer.get_feature_names_out()]

#refresh memory to not crash
del sample
gc.collect()

#streaming to create
first_chunk = True

for chunk in pd.read_csv(INPUT_FILE, chunksize=CHUNK_SIZE):

    tfidf_sparse = vectorizer.transform(chunk['review'].fillna(''))
    tfidf_df = pd.DataFrame(tfidf_sparse.toarray(), columns=tfidf_cols)
    metadata = chunk.drop(columns=['review']).reset_index(drop=True)
    combined = pd.concat([metadata, tfidf_df], axis=1)
    combined.to_csv(OUTPUT_FILE, mode='a' if not first_chunk else 'w', header=first_chunk, index=False)
    first_chunk = False

print(f"TF-IDF Feature CSV created at: {OUTPUT_FILE}")
